# Структура меток (аннотаций) EDF-файлов
Датасет мысленного произнесения слов — два языковых набора: **Russian** и **Spanish**.

In [ ]:
import mne
import pandas as pd
from pathlib import Path

mne.set_log_level('WARNING')

BASE = Path(r'D:\Inner Speech Dataset\Inner Speech Dataset')

In [ ]:
def annotations_summary(edf_path):
    raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
    ann = raw.annotations
    return pd.DataFrame({
        'onset':       ann.onset,
        'duration':    ann.duration,
        'description': ann.description,
    })


def show_label_structure(language):
    lang_dir = BASE / language
    edf_files = sorted(lang_dir.rglob('*.edf'))
    print(f'=== {language} ({len(edf_files)} subjects) ===')
    all_dfs = []
    for edf in edf_files:
        subject = edf.parent.name
        df = annotations_summary(edf)
        df.insert(0, 'subject', subject)
        df.insert(1, 'language', language)
        all_dfs.append(df)
        counts = df['description'].value_counts().to_dict()
        print(f'  {subject}: {len(df)} ann | labels: {sorted(df["description"].unique())}')
        print(f'    counts: {counts}')
    return pd.concat(all_dfs, ignore_index=True)

In [ ]:
df_ru = show_label_structure('Russian')

In [ ]:
df_es = show_label_structure('Spanish')

## Сводная таблица: количество аннотаций по меткам

## Выявление участников с нетипичной методологией (Russian)

In [ ]:
import numpy as np

def analyze_methodology(language='Russian'):
    lang_dir = BASE / language
    edf_files = sorted(lang_dir.rglob('*.edf'))

    rows = []
    for edf in edf_files:
        subject = edf.parent.name
        raw = mne.io.read_raw_edf(edf, preload=False, verbose=False)
        ann = raw.annotations

        onsets = ann.onset
        n_ann = len(onsets)

        if n_ann < 2:
            rows.append({'subject': subject, 'n_annotations': n_ann,
                         'median_gap_s': None, 'max_gap_s': None, 'flag': 'too few'})
            continue

        gaps = np.diff(onsets)  # интервалы между соседними аннотациями
        rows.append({
            'subject':       subject,
            'n_annotations': n_ann,
            'median_gap_s':  round(float(np.median(gaps)), 2),
            'max_gap_s':     round(float(np.max(gaps)), 2),
        })

    df = pd.DataFrame(rows).sort_values('subject')

    # Типичная методология: 1 метка = 1 произнесение → много меток, короткие равномерные интервалы
    # Нетипичная: 1 метка = N произнесений → мало меток, большие/неравномерные интервалы
    median_n = df['n_annotations'].median()
    df['flag'] = df['n_annotations'].apply(
        lambda x: '⚠ нетипичный' if x < median_n * 0.5 else 'норма'
    )

    return df

df_method = analyze_methodology('Russian')
display(df_method)

print('\nПодозрительные участники:')
display(df_method[df_method['flag'] != 'норма'])

In [ ]:
df_all = pd.concat([df_ru, df_es], ignore_index=True)

pivot = (
    df_all
    .groupby(['language', 'subject', 'description'])
    .size()
    .reset_index(name='count')
    .pivot_table(index=['language', 'subject'], columns='description', values='count', fill_value=0)
)
display(pivot)

In [ ]:
print('Все уникальные метки по языкам:')
for lang, grp in df_all.groupby('language'):
    print(f'  {lang}: {sorted(grp["description"].unique())}')

---\n## Информация по эпохам (.fif файлы)

In [ ]:
def epochs_info(language):
    lang_dir = BASE / language
    fif_files = sorted(lang_dir.rglob('*-epo.fif'))

    rows = []
    for fif in fif_files:
        subject = fif.parent.name
        epochs = mne.read_epochs(fif, preload=False, verbose=False)

        n_trials   = len(epochs)
        n_channels = len(epochs.ch_names)
        sfreq      = epochs.info['sfreq']
        tmin       = round(epochs.tmin, 4)
        tmax       = round(epochs.tmax, 4)
        epoch_len  = round(tmax - tmin, 4)
        n_times    = epochs.times.shape[0]

        event_counts = {k: v for k, v in zip(
            epochs.event_id.keys(),
            [(epochs.events[:, 2] == v).sum() for v in epochs.event_id.values()]
        )}

        rows.append({
            'language':    language,
            'subject':     subject,
            'n_trials':    n_trials,
            'n_channels':  n_channels,
            'sfreq_Hz':    sfreq,
            'tmin_s':      tmin,
            'tmax_s':      tmax,
            'epoch_len_s': epoch_len,
            'n_timepoints':n_times,
            'event_id':    epochs.event_id,
            'trial_counts':event_counts,
        })

    return pd.DataFrame(rows)


df_epo_ru = epochs_info('Russian')
df_epo_es = epochs_info('Spanish')
df_epo_all = pd.concat([df_epo_ru, df_epo_es], ignore_index=True)

In [ ]:
display(df_epo_all[['language','subject','n_trials','n_channels','sfreq_Hz','tmin_s','tmax_s','epoch_len_s','n_timepoints']])

In [ ]:
# Количество трайлов по словам — сводная таблица
trial_df = pd.DataFrame(df_epo_all['trial_counts'].tolist())
trial_df.insert(0, 'subject', df_epo_all['subject'].values)
trial_df.insert(0, 'language', df_epo_all['language'].values)
trial_df = trial_df.set_index(['language', 'subject'])
display(trial_df)

## Идентификация EMG каналов

In [ ]:
EMG_CHANNELS = {'B1+', 'B1-', 'B2+', 'B2-', 'EMG1', 'EMG2', 'Pharing1', 'Pharing2'}

def show_channel_names(language):
    lang_dir = BASE / language
    edf_files = sorted(lang_dir.rglob('*.edf'))

    rows = []
    for edf in edf_files:
        subject = edf.parent.name
        raw = mne.io.read_raw_edf(edf, preload=False, verbose=False)
        ch_names = raw.ch_names
        emg_present = [ch for ch in ch_names if ch in EMG_CHANNELS]
        eeg_channels = [ch for ch in ch_names if ch not in EMG_CHANNELS]
        rows.append({
            'language':    language,
            'subject':     subject,
            'n_total':     len(ch_names),
            'n_eeg':       len(eeg_channels),
            'n_emg':       len(emg_present),
            'emg_channels': emg_present if emg_present else None,
        })

    return pd.DataFrame(rows)


df_ch_ru = show_channel_names('Russian')
df_ch_es = show_channel_names('Spanish')
df_ch_all = pd.concat([df_ch_ru, df_ch_es], ignore_index=True)

display(df_ch_all.set_index(['language', 'subject']))

In [ ]:
def check_emg_in_fif(language):
    lang_dir = BASE / language
    fif_files = sorted(lang_dir.rglob('*-epo.fif'))

    rows = []
    for fif in fif_files:
        subject = fif.parent.name
        epochs = mne.read_epochs(fif, preload=False, verbose=False)

        ch_types = {ch: mne.channel_type(epochs.info, i)
                    for i, ch in enumerate(epochs.ch_names)}

        emg_info = {ch: ch_types[ch] for ch in epochs.ch_names if ch in EMG_CHANNELS}

        rows.append({
            'language':    language,
            'subject':     subject,
            'n_total':     len(epochs.ch_names),
            'emg_found':   emg_info if emg_info else None,
        })

    return pd.DataFrame(rows)


fif_emg_ru = check_emg_in_fif('Russian')
fif_emg_es = check_emg_in_fif('Spanish')
fif_emg_all = pd.concat([fif_emg_ru, fif_emg_es], ignore_index=True)

print('EMG каналы в fif (имя → тип):')
display(fif_emg_all.set_index(['language', 'subject']))

In [ ]:
def show_fif_channels(language):
    lang_dir = BASE / language
    fif_files = sorted(lang_dir.rglob('*-epo.fif'))

    for fif in fif_files:
        subject = fif.parent.name
        epochs = mne.read_epochs(fif, preload=False, verbose=False)
        print(f"[{language}] {subject} ({len(epochs.ch_names)} каналов):")
        print(f"  {epochs.ch_names}")
        print()

show_fif_channels('Russian')
show_fif_channels('Spanish')

## Препроцессинг в fif-файлах

In [ ]:
def show_preprocessing(language):
    lang_dir = BASE / language
    fif_files = sorted(lang_dir.rglob('*-epo.fif'))

    for fif in fif_files:
        subject = fif.parent.name
        epochs = mne.read_epochs(fif, preload=False, verbose=False)
        info = epochs.info

        print(f"[{language}] {subject}")
        print(f"  Фильтры : highpass={info['highpass']} Hz, lowpass={info['lowpass']} Hz")
        print(f"  Baseline: {epochs.baseline}")
        print(f"  Reject  : {epochs.reject}")
        print(f"  Flat    : {epochs.flat}")

        # SSP проекторы
        projs = info.get('projs', [])
        if projs:
            print(f"  Проекторы (SSP): {[p['desc'] for p in projs]}")
        else:
            print(f"  Проекторы (SSP): нет")

        # История обработки
        proc = info.get('proc_history', [])
        if proc:
            for step in proc:
                creator = step.get('creator', '?')
                date    = step.get('date', '?')
                print(f"  proc_history: {creator} @ {date}")
                maxinfo = step.get('max_info', {})
                if maxinfo:
                    print(f"    max_info: {maxinfo}")
        else:
            print(f"  proc_history: нет")

        print()

show_preprocessing('Russian')
show_preprocessing('Spanish')

In [ ]:
# Ищем ICA-файлы рядом с данными
print('=== Поиск *-ica.fif файлов ===')
for language in ('Russian', 'Spanish'):
    ica_files = sorted((BASE / language).rglob('*ica*'))
    if ica_files:
        for f in ica_files:
            print(f'  {f}')
    else:
        print(f'  [{language}] ICA-файлов не найдено')

print()

# Смотрим description и комментарии внутри fif
print('=== info[description] и info[experimenter] в fif ===')
for language in ('Russian', 'Spanish'):
    for fif in sorted((BASE / language).rglob('*-epo.fif')):
        subject = fif.parent.name
        epochs = mne.read_epochs(fif, preload=False, verbose=False)
        info = epochs.info
        desc  = info.get('description', None)
        exp   = info.get('experimenter', None)
        notes = [f'description={desc!r}' if desc else None,
                 f'experimenter={exp!r}' if exp else None]
        notes = [n for n in notes if n]
        print(f'  [{language}] {subject}: {", ".join(notes) if notes else "нет мета-комментариев"}')

## Препроцессинг в EDF-файлах

In [ ]:
def show_edf_preprocessing(language):
    lang_dir = BASE / language
    edf_files = sorted(lang_dir.rglob('*.edf'))

    for edf in edf_files:
        subject = edf.parent.name
        raw = mne.io.read_raw_edf(edf, preload=False, verbose=False)
        info = raw.info

        print(f"[{language}] {subject}")
        print(f"  Фильтры     : highpass={info['highpass']} Hz, lowpass={info['lowpass']} Hz")
        print(f"  Доп. инфо   : description={info.get('description', None)!r}")
        print(f"  experimenter: {info.get('experimenter', None)!r}")

        # EDF-заголовок доступен через _raw_extras
        extras = getattr(raw, '_raw_extras', [{}])
        if extras:
            ex = extras[0]
            print(f"  EDF subtype      : {ex.get('subtype', '?')}")
            print(f"  EDF subject_info : {ex.get('subject_info', '?')}")
            print(f"  EDF recording    : {ex.get('rec_id', '?')}")
            prefilter = ex.get('prefilter', None)
            if not prefilter:
                # попробуем достать из сигналов
                prefilter = [s.get('prefilter', '') for s in ex.get('signals', [])]
                prefilter = list(dict.fromkeys(filter(None, prefilter)))  # уникальные
            print(f"  Prefilter field  : {prefilter}")

        proc = info.get('proc_history', [])
        if proc:
            for step in proc:
                print(f"  proc_history: {step.get('creator','?')} @ {step.get('date','?')}")
        else:
            print(f"  proc_history: нет")

        print()

show_edf_preprocessing('Russian')
show_edf_preprocessing('Spanish')